# Generate Pipeline (Recipe → Molecule Graph/Analysis)

이 노트북은 `vocab_alignment.ipynb`, `generate/recipe_molecule_pipeline_redesign.ipynb`, `analysis/cuisine_only_cluster_analysis.ipynb`의 핵심 로직을 정리한 통합 버전입니다.

- 입력: `../recipes.csv` (기존 `_with_ratio` 파일 대체), `../flavordb_alias_in_vocab_only.csv`, `../molecules.csv`
- 출력: cuisine별/ALL graph csv, clustering/lift plot, cluster graph 시각화
- 핵심 수식:

$$
S(r,m)=\sum_{i \in I(r)}
w_{r,i}\cdot\frac{\mathbf{1}\{m \in M(i)\}}{|M(i)|}
$$


## 0) 환경 설정

- 진행 시간이 긴 반복문에는 `tqdm`을 적용했습니다.
- 중간 산출물은 `head(10)`, row 수(`shape`)를 확인하도록 출력합니다.


In [1]:
from __future__ import annotations

import ast
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

plt.style.use("seaborn-v0_8-whitegrid")


/home/samyang/miniconda3/envs/gfn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Paths
RECIPES_PATH = Path("../recipes.csv")
FLAVORDB_PATH = Path("../preprocess/flavordb_alias_in_vocab_only.csv")
MOLECULES_PATH = Path("../molecules.csv")

OUT_DIR = Path("../result")
GRAPH_DIR = OUT_DIR / "graph"
ANALYSIS_DIR = OUT_DIR / "analysis"
PLOT_DIR = OUT_DIR / "plots"

for d in [OUT_DIR, GRAPH_DIR, ANALYSIS_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output directories:")
for d in [GRAPH_DIR, ANALYSIS_DIR, PLOT_DIR]:
    print(" -", d.resolve())


Output directories:
 - /home/samyang/CompoundAnalysis/result/graph
 - /home/samyang/CompoundAnalysis/result/analysis
 - /home/samyang/CompoundAnalysis/result/plots


## 1) 로드 및 기본 유틸


In [3]:
def norm_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s']", "", s)
    return s.strip()


def find_col(df: pd.DataFrame, candidates: list[str]) -> str:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise KeyError(f"Not found among candidates={candidates}. Available columns={list(df.columns)}")


def parse_ingredient_ratio(raw):
    """
    허용 포맷 예시:
    - list[dict]: [{'ingredient':'onion','ratio':0.2}, ...]
    - dict[str,float]: {'onion':0.2, ...}
    - str literal of above
    """
    if pd.isna(raw):
        return []
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return []
        try:
            raw = ast.literal_eval(raw)
        except Exception:
            return []

    out = []
    if isinstance(raw, dict):
        for k, v in raw.items():
            out.append((norm_text(k), float(v)))
    elif isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                ing = item.get("ingredient", item.get("name", item.get("ing")))
                ratio = item.get("ratio", item.get("weight", item.get("w", 0)))
                if ing is not None:
                    out.append((norm_text(ing), float(ratio)))
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append((norm_text(item[0]), float(item[1])))
    return [(i, w) for i, w in out if i and w > 0]


In [6]:
recipes = pd.read_csv(RECIPES_PATH)
flavordb = pd.read_csv(FLAVORDB_PATH)
molecules = pd.read_csv(MOLECULES_PATH)

print("recipes:", recipes.shape)
print("flavordb:", flavordb.shape)
print("molecules:", molecules.shape)

display(recipes.head(10))
display(flavordb.head(10))
display(molecules.head(10))

recipes: (554161, 4)
flavordb: (857, 7)
molecules: (1788, 4)


,name,cleaned_ingredients,ingredients_ratio,cuisine
0,Worlds Best Mac and Cheese,"[['penne', 170.0], ['cheddar cheese', 28.0], [...","[['penne', 112.137], ['cheddar cheese', 18.47]...",Italian
1,Dilly Macaroni Salad Recipe,"[['macaroni', 140.0], ['cheese', 113.0], ['cel...","[['macaroni', 237.893], ['cheese', 192.014], [...",American
2,Gazpacho,"[['tomato', 1600.0], ['salt', 3.0], ['onion', ...","[['tomato', 651.466], ['salt', 1.221], ['onion...",French
3,Kombu Tea Grilled Chicken Thigh,"[['chicken thigh', 200.0], ['kombu', 4.0], ['w...","[['chicken thigh', 975.61], ['kombu', 19.512],...",Korean
4,Zucchini Nut Bread,"[['flour', 240.0], ['cinnamon', 8.0], ['baking...","[['flour', 167.744], ['cinnamon', 5.591], ['ba...",American
5,Salmon & Salad a La SPORTZ,"[['green onion', 75.0], ['pepper', 150.0], ['s...","[['green onion', 36.819], ['pepper', 73.638], ...",American
6,Brown Sugar 'Karintou' Snacks,"[['flour', 200.0], ['baking powder', 4.0], ['s...","[['flour', 382.958], ['baking powder', 7.659],...",Fusion
7,Corn Casserole,"[['tomato', 240.0], ['corn', 240.0], ['cheddar...","[['tomato', 164.271], ['corn', 164.271], ['che...",Mexican
8,Broccoli Rice Bake,"[['broccoli', 283.5], ['rice', 185.0], ['chedd...","[['broccoli', 317.167], ['rice', 206.97], ['ch...",American
9,Steak & Asparagus Wraps,"[['marinade', 120.0], ['beef sirloin', 453.6],...","[['marinade', 56.775], ['beef sirloin', 214.61...",American


,Unnamed: 0,entity id,alias,synonyms,scientific name,category,molecules
0,0,1,bread,{'bakery products'},poacceae,bakery,"{27457, 31252, 7976, 22201, 26331, 26808}"
1,1,2,bread,{'bread'},poacceae,bakery,"{1031, 1032, 644104, 527, 8723, 31260, 15394, ..."
2,2,3,rye bread,{'rye bread'},rye,bakery,"{32065, 644104, 72, 18635, 460, 332, 12366, 89..."
3,3,4,bread,"{'soda scones', 'soda farls'}",wheat,bakery,"{30914, 6915, 5365891, 12170, 14286, 8082, 312..."
4,4,5,white bread,{'white bread'},wheat,bakery,"{7361, 994, 10883, 7362, 11173, 5365891, 11559..."
5,5,6,whole wheat flour,{'wholewheat bread'},wheat,bakery,"{107905, 8194, 10883, 13187, 5283329, 5283335,..."
6,6,7,wort,{'wort'},barley,beverage,"{13187, 9862, 135, 18827, 7824, 61712, 19602, ..."
7,7,8,arrack,{'arak'},grape,beverage alcoholic,"{240, 31249, 1031, 6584, 7997}"
8,8,9,beer,{'beer'},poacceae,beverage alcoholic,"{62465, 8194, 8193, 1031, 644104, 1032, 62484,..."
9,9,10,beer,"{'malwa', 'pombe', 'millet beer', 'opaque beer...",eragrostideae,beverage alcoholic,"{6560, 8038, 7654, 7147, 1068, 14286, 527, 240..."


,Unnamed: 0,pubchem id,common name,flavor profile
0,0,4,1-Aminopropan-2-ol,{'fishy'}
1,1,49,3-Methyl-2-oxobutanoic acid,{'fruity'}
2,2,58,2-oxobutanoic acid,"{'brown', 'creamy', 'sweet', 'lactonic', 'cara..."
3,3,70,4-Methyl-2-oxovaleric acid,{'fruity'}
4,4,72,"3,4-Dihydroxybenzoic Acid","{'phenolic', 'balsamic', 'mild'}"
5,5,107,3-Phenylpropanoic acid,"{'fatty', 'cinnamon', 'sweet', 'balsamic', 'ro..."
6,6,125,4-hydroxybenzyl alcohol,"{'fruity', 'sweet', 'almond', 'coconut', 'bitt..."
7,7,126,4-hydroxybenzaldehyde,"{'woody', 'nutty', 'balsam', 'sweet', 'almond'}"
8,8,135,4-Hydroxybenzoic Acid,"{'phenolic', 'nutty'}"
9,9,176,acetic acid,"{'vinegar', 'pungent', 'sharp', 'sour'}"


In [5]:
# Column detection
recipe_id_col = find_col(recipes, ["recipe_id", "id", "rid"])
cuisine_col = find_col(recipes, ["cuisine", "cuisine_name"])

# recipes.csv에 ratio 컬럼이 문자열로 저장된 케이스를 기본으로 처리
ratio_col = find_col(recipes, ["ingredient_ratio", "ingredients_with_ratio", "ingredients_ratio", "ingredients"])

ing_col = find_col(flavordb, ["ingredient", "entity_alias_readable", "entity_alias", "name"])

mol_col = find_col(flavordb, ["molecule id", "molecule_id", "pubchem id", "pubchem_id"])

print("recipe_id_col:", recipe_id_col)
print("cuisine_col:", cuisine_col)
print("ratio_col:", ratio_col)
print("flavor ingredient col:", ing_col)
print("flavor molecule col:", mol_col)


KeyError: "Not found among candidates=['recipe_id', 'id', 'rid']. Available columns=['name', 'cleaned_ingredients', 'ingredients_ratio', 'cuisine']"

## 2) FlavorDB ingredient → molecule set $M(i)$ 생성


In [ ]:
flavordb["ingredient_norm"] = flavordb[ing_col].map(norm_text)
flavordb["molecule_id"] = pd.to_numeric(flavordb[mol_col], errors="coerce")

ing_to_molecules = (
    flavordb.dropna(subset=["ingredient_norm", "molecule_id"])
            .groupby("ingredient_norm")["molecule_id"]
            .apply(lambda x: sorted(set(int(v) for v in x if pd.notna(v))))
            .to_dict()
)

print("mapped ingredients:", len(ing_to_molecules))
print("sample:", list(ing_to_molecules.items())[:5])


## 3) Recipe long format 생성 (전처리 결과 사용)


In [ ]:
records = []
for _, row in tqdm(recipes.iterrows(), total=len(recipes), desc="Parse recipes"):
    rid = row[recipe_id_col]
    cuisine = row[cuisine_col]
    pairs = parse_ingredient_ratio(row[ratio_col])

    z = sum(w for _, w in pairs)
    if z <= 0:
        continue

    for ing, w in pairs:
        records.append({
            "recipe_id": int(rid),
            "cuisine": str(cuisine),
            "ingredient": ing,
            "w_ri": float(w / z),
        })

recipes_long = pd.DataFrame(records)
print("recipes_long:", recipes_long.shape)
display(recipes_long.head(10))

recipes_long.to_csv(OUT_DIR / "recipes_long_normalized.csv", index=False)


## 4) 핵심 수식으로 recipe-molecule score 계산

각 레시피 $r$와 분자 $m$에 대해 아래를 계산합니다.

$$
S(r,m)=\sum_{i \in I(r)} w_{r,i}\cdot\frac{\mathbf{1}\{m \in M(i)\}}{|M(i)|}
$$


In [ ]:
score_rows = []
unk_rows = []

for rid, grp in tqdm(recipes_long.groupby("recipe_id"), desc="Compute S(r,m)"):
    acc = defaultdict(float)
    mass_covered = 0.0

    for ing, w in grp[["ingredient", "w_ri"]].itertuples(index=False):
        mols = ing_to_molecules.get(ing, [])
        if not mols:
            continue
        contrib = w / len(mols)
        for m in mols:
            acc[m] += contrib
        mass_covered += w

    for m, s in acc.items():
        score_rows.append({"recipe_id": int(rid), "molecule_id": int(m), "S_rm": float(s)})

    unk_rows.append({"recipe_id": int(rid), "u_r": float(max(0.0, 1.0 - mass_covered))})

recipe_molecule = pd.DataFrame(score_rows)
recipe_unk = pd.DataFrame(unk_rows)

print("recipe_molecule:", recipe_molecule.shape)
print("recipe_unk:", recipe_unk.shape)
display(recipe_molecule.head(10))
display(recipe_unk.head(10))

recipe_molecule.to_csv(GRAPH_DIR / "recipe_molecule_edges.csv", index=False)
recipe_unk.to_csv(GRAPH_DIR / "recipe_unk_mass.csv", index=False)


## 5) Cuisine별 + ALL graph csv 저장


In [ ]:
rid_to_cuisine = recipes_long[["recipe_id", "cuisine"]].drop_duplicates().set_index("recipe_id")["cuisine"].to_dict()
recipe_molecule["cuisine"] = recipe_molecule["recipe_id"].map(rid_to_cuisine)

def export_graph_for_group(df_edges: pd.DataFrame, key: str):
    mol_weight = (
        df_edges.groupby("molecule_id", as_index=False)["S_rm"].sum()
               .rename(columns={"S_rm": "weight"})
    )
    out_edges = GRAPH_DIR / f"{key}_recipe_molecule_edges.csv"
    out_nodes = GRAPH_DIR / f"{key}_molecule_nodes.csv"
    df_edges.to_csv(out_edges, index=False)
    mol_weight.to_csv(out_nodes, index=False)
    print(f"[{key}] edges={len(df_edges):,}, nodes={len(mol_weight):,}")
    display(df_edges.head(10))
    display(mol_weight.head(10))

for cuisine, sub in tqdm(recipe_molecule.groupby("cuisine"), desc="Export cuisine graphs"):
    export_graph_for_group(sub, re.sub(r"\W+", "_", str(cuisine)).strip("_") or "unknown")

export_graph_for_group(recipe_molecule, "ALL")


## 6) Molecule co-occurrence graph + clustering + lift (Cuisine별/ALL)


In [ ]:
def build_recipe_to_molset(df_edges: pd.DataFrame):
    d = defaultdict(set)
    for rid, mid in df_edges[["recipe_id", "molecule_id"]].itertuples(index=False):
        d[int(rid)].add(int(mid))
    return d


def build_molecule_graph(rids, rid2molset, min_cooc=3):
    cooc = Counter()
    for rid in rids:
        mols = sorted(rid2molset.get(int(rid), []))
        for i in range(len(mols)):
            a = mols[i]
            for j in range(i + 1, len(mols)):
                b = mols[j]
                cooc[(a, b)] += 1

    G = nx.Graph()
    for (a, b), w in cooc.items():
        if w >= min_cooc:
            G.add_edge(a, b, weight=w)
    return G


def cluster_graph(G: nx.Graph):
    if G.number_of_nodes() == 0:
        return {}
    communities = nx.algorithms.community.louvain_communities(G, weight="weight", seed=42)
    part = {}
    for cid, comm in enumerate(communities):
        for node in comm:
            part[node] = cid
    return part


def ingredient_lift(recipes_subset, ingredient_col="ingredient", target_flag_col="in_cluster"):
    # P(ingredient | cluster) / P(ingredient | cuisine)
    base = recipes_subset.groupby(ingredient_col).size().rename("base_n")
    sub = recipes_subset[recipes_subset[target_flag_col] == 1].groupby(ingredient_col).size().rename("cluster_n")
    m = pd.concat([base, sub], axis=1).fillna(0)

    N = len(recipes_subset)
    K = max(1, int((recipes_subset[target_flag_col] == 1).sum()))
    m["p_base"] = m["base_n"] / max(1, N)
    m["p_cluster"] = m["cluster_n"] / K
    m["lift"] = m["p_cluster"] / m["p_base"].replace(0, np.nan)
    return m.sort_values("lift", ascending=False)


In [ ]:
rid2molset = build_recipe_to_molset(recipe_molecule)
rid2ings = recipes_long.groupby("recipe_id")["ingredient"].apply(list).to_dict()
all_cuisines = sorted(recipes_long["cuisine"].dropna().unique().tolist())
target_groups = ["ALL"] + all_cuisines

analysis_summary = []

for group in tqdm(target_groups, desc="Cuisine clustering"):
    if group == "ALL":
        rids = sorted(recipes_long["recipe_id"].unique())
    else:
        rids = sorted(recipes_long.loc[recipes_long["cuisine"] == group, "recipe_id"].unique())

    print(f"
==== {group} ====")
    print("recipes:", len(rids))

    G = build_molecule_graph(rids, rid2molset, min_cooc=3)
    part = cluster_graph(G)

    print("graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges(), "clusters:", len(set(part.values())) if part else 0)

    # Cluster graph visualize 저장
    fig = plt.figure(figsize=(8, 6))
    if G.number_of_nodes() > 0:
        pos = nx.spring_layout(G, seed=42)
        node_colors = [part.get(n, -1) for n in G.nodes()]
        nx.draw_networkx(
            G,
            pos=pos,
            node_size=40,
            with_labels=False,
            node_color=node_colors,
            cmap="tab20",
            edge_color="gray",
            alpha=0.8,
        )
        plt.title(f"{group} molecule cluster graph")
    else:
        plt.text(0.5, 0.5, "Empty graph", ha="center", va="center")
        plt.title(f"{group} molecule cluster graph")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{group}_cluster_graph.png", dpi=150)
    plt.close(fig)

    # recipe dominant cluster (hard assign)
    rid_rows = []
    for rid in rids:
        cnt = Counter(part.get(m) for m in rid2molset.get(rid, set()) if m in part)
        cnt.pop(None, None)
        cid = cnt.most_common(1)[0][0] if cnt else -1
        rid_rows.append((rid, cid))
    rid_cluster = pd.DataFrame(rid_rows, columns=["recipe_id", "cluster_id"])

    # lift 계산용 long
    tmp = recipes_long[recipes_long["recipe_id"].isin(rids)][["recipe_id", "ingredient"]].copy()
    tmp = tmp.merge(rid_cluster, on="recipe_id", how="left")

    lift_records = []
    for cid in sorted(c for c in rid_cluster["cluster_id"].unique() if c >= 0):
        tmp["in_cluster"] = (tmp["cluster_id"] == cid).astype(int)
        lf = ingredient_lift(tmp, ingredient_col="ingredient", target_flag_col="in_cluster").head(20).reset_index()
        lf.insert(0, "group", group)
        lf.insert(1, "cluster_id", cid)
        lift_records.append(lf)

    if lift_records:
        lift_df = pd.concat(lift_records, ignore_index=True)
    else:
        lift_df = pd.DataFrame(columns=["group", "cluster_id", "ingredient", "base_n", "cluster_n", "p_base", "p_cluster", "lift"])

    lift_path = ANALYSIS_DIR / f"{group}_cluster_lift_top20.csv"
    lift_df.to_csv(lift_path, index=False)

    # cluster 비중 bar plot
    fig = plt.figure(figsize=(7, 4))
    dist = rid_cluster["cluster_id"].value_counts().sort_index()
    dist = dist[dist.index >= 0]
    if len(dist):
        dist.plot(kind="bar")
        plt.ylabel("# recipes")
        plt.title(f"{group} cluster distribution")
    else:
        plt.text(0.5, 0.5, "No cluster assignment", ha="center", va="center")
        plt.title(f"{group} cluster distribution")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"{group}_cluster_distribution.png", dpi=150)
    plt.close(fig)

    print("lift rows:", len(lift_df))
    display(lift_df.head(10))

    analysis_summary.append({
        "group": group,
        "n_recipes": len(rids),
        "n_graph_nodes": G.number_of_nodes(),
        "n_graph_edges": G.number_of_edges(),
        "n_clusters": len(set(part.values())) if part else 0,
        "lift_rows": len(lift_df),
    })


## 7) 요약 저장


In [ ]:
summary_df = pd.DataFrame(analysis_summary)
summary_df.to_csv(ANALYSIS_DIR / "analysis_summary.csv", index=False)

print("analysis summary:", summary_df.shape)
display(summary_df.head(10))

elapsed_msg = "Notebook cells completed. Check result/graph, result/analysis, result/plots"
print(elapsed_msg)
